# Plant Leaf Disease Detection Using CNN
**Deep Learning project for Google Colab**

This project classifies plant leaf images into disease/healthy categories using a Convolutional Neural Network (CNN), following a workflow similar to a traffic-sign recognition notebook: image loading, resizing, normalization, CNN training, evaluation, and prediction.

**Dataset:** [PlantVillage / Plant Disease dataset on Kaggle](https://www.kaggle.com/datasets/emmarex/plantdisease)

> Note: Dataset folder layouts can vary. The code below searches for the folder that contains the class subfolders. Model performance on clean dataset images may not generalize perfectly to real-world farm photos.


In [ ]:
# Cell 1 — Install the dataset downloader
!pip -q install kagglehub scikit-learn seaborn


In [ ]:
# Cell 2 — Download the dataset
import os
from pathlib import Path
import kagglehub

dataset_path = Path(kagglehub.dataset_download("emmarex/plantdisease"))
print("Downloaded to:", dataset_path)


In [ ]:
# Cell 3 — Find the directory containing one subfolder per class
# This handles common variations in the dataset's folder structure.
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    try:
        return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    except Exception:
        return 0

candidates = []
for p in [dataset_path] + [p for p in dataset_path.rglob("*") if p.is_dir()]:
    try:
        class_dirs = [d for d in p.iterdir() if d.is_dir() and count_images(d) > 0]
        if len(class_dirs) >= 2:
            candidates.append((len(class_dirs), count_images(p), p))
    except (PermissionError, OSError):
        pass

if not candidates:
    raise RuntimeError(
        "Could not find a folder with class subfolders. Inspect dataset_path with "
        "`!find {dataset_path} -maxdepth 3 -type d | head -80` and update data_root."
    )

# Prefer the folder with the most class folders, then the most images.
candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
data_root = candidates[0][2]
class_dirs = sorted([d for d in data_root.iterdir() if d.is_dir() and count_images(d) > 0])

print("Image folder:", data_root)
print("Number of classes:", len(class_dirs))
print("Classes:", [d.name for d in class_dirs])


In [ ]:
# Cell 4 — Build image/label lists and create train, validation, and test splits
import numpy as np
from sklearn.model_selection import train_test_split

class_names = [d.name for d in class_dirs]
class_to_id = {name: i for i, name in enumerate(class_names)}

file_paths, labels = [], []
for class_dir in class_dirs:
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            file_paths.append(str(p))
            labels.append(class_to_id[class_dir.name])

file_paths = np.array(file_paths)
labels = np.array(labels, dtype=np.int32)

# 70% train, 15% validation, 15% test (stratified by class)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    file_paths, labels, test_size=0.30, random_state=42, stratify=labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print(f"Total images: {len(file_paths):,}")
print(f"Train: {len(train_paths):,} | Validation: {len(val_paths):,} | Test: {len(test_paths):,}")
print(f"Classes: {len(class_names)}")


In [ ]:
# Cell 5 — Create efficient TensorFlow input pipelines
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)

# Show a sample batch
images, image_labels = next(iter(train_ds))
plt.figure(figsize=(10, 8))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy())
    plt.title(class_names[int(image_labels[i])], fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 6 — Build the CNN model
from tensorflow.keras import layers, models

num_classes = len(class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
], name="data_augmentation")

model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    data_augmentation,

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.40),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# Cell 7 — Train the model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint("best_plant_disease_cnn.keras", monitor="val_accuracy",
                    save_best_only=True, verbose=1)
]

EPOCHS = 20  # Increase if needed; GPU runtime is recommended.
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


In [ ]:
# Cell 8 — Plot accuracy and loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Cell 9 — Evaluate on the held-out test set
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


In [ ]:
# Cell 10 — Classification report and confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
probabilities = model.predict(test_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

print(classification_report(
    y_true, y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
fig, ax = plt.subplots(figsize=(max(10, len(class_names) * 0.45), max(8, len(class_names) * 0.40)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=90, colorbar=False, values_format="d")
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 11 — Predict one uploaded leaf image
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()

for filename, file_bytes in uploaded.items():
    pil_image = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    display(pil_image.resize((300, 300)))

    resized = pil_image.resize(IMG_SIZE)
    arr = np.array(resized).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = model.predict(arr, verbose=0)[0]
    predicted_id = int(np.argmax(probs))

    print("Image:", filename)
    print("Predicted class:", class_names[predicted_id])
    print(f"Confidence: {probs[predicted_id] * 100:.2f}%")

    # Show top 5 predictions
    top_ids = np.argsort(probs)[-5:][::-1]
    print("\nTop predictions:")
    for idx in top_ids:
        print(f"{class_names[int(idx)]}: {probs[idx] * 100:.2f}%")


In [ ]:
# Cell 12 — Save the trained model and class names
import json

model.save("plant_leaf_disease_cnn.keras")
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("Saved: plant_leaf_disease_cnn.keras and class_names.json")

# Download model file to your computer (optional)
from google.colab import files
files.download("plant_leaf_disease_cnn.keras")
files.download("class_names.json")
